# Prompt Engineering

In [1]:
%pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 40.7 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [boto3]32m4/5 [boto3]re]
Note: you may need to restart the kernel to use updated packages.


# Requirements
https://catalog.us-east-1.prod.workshops.aws/workshops/c6b88897-84a7-4885-b9f0-855e2fc61378/en-US/05-prerequisites/02-self-paced

- **Request model access:** AWS account > Amazon Bedrock > Model access
- **Create IAM Role for Bedrock** (and other services...): see example IAM policy to grant access to Bedrock APIs 
- **Python v3.8+**

In [5]:
import json
import os
import sys

import boto3
import botocore

## Set AWS Bedrock client
- region

In [ ]:
# Choose region from environment (AWS_REGION or AWS_DEFAULT_REGION) or fallback to a default
region = os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION") or "us-west-2"

# Option 1 — pass region_name directly to the client (explicit):
# bedrock = boto3.client("bedrock-runtime", region_name="us-west-2")

# Option 2 — use a Session (useful if you want multiple clients with the same region/creds):
# session = boto3.Session(region_name="us-west-2")
# bedrock = session.client("bedrock-runtime")

# Option 3 — use the environment-based region variable (recommended for flexible scripts):
bedrock_client = boto3.client("bedrock-runtime", region_name=region)

# Print which region is being used (helpful for debugging):
print("Using AWS region:", region)

Using AWS region: us-west-2


# Description 
https://catalog.us-east-1.prod.workshops.aws/amazon-bedrock/en-US/050-prompt-engineering

### Patterns
- Zero shot
- Few Shot
- CoT

### Steps
1. **Create prompt_data** for a given pattern
2. **Call Invoke Model API**: post request to get a response
    Request => headers + body (includes prompt_data)
    - headers: modelId + accept + contentType
    - body (json): \<model_variables> 
3. **Read body** (json) from response
4. **Extract answer** from body (path will vary depending on model used)
    - Zero-shot: body.completion
    - Few-shot: body.completion
    - CoT: body.content.text

# Patterns

## Zero Shot
https://github.com/ExamProCo/AWS-Examples/blob/main/bedrock/prompt-engineering/zero-shot.ipynb

- model: anthropic claude2

In [9]:
prompt_data = """
Human: 
Sulfuric acid reacts with sodium chloride, and gives <chemical1>_____</chemical1> and <chemical2>_____</chemical2>:
Assistant: the chemical1 and chemical2 are:
"""

In [ ]:
body = json.dumps(
    {
        "prompt": prompt_data,
        "max_tokens_to_sample": 200,
        "temperature": 1.0,
    }
)

Invoke the model to get response

In [ ]:
response = bedrock_client.invoke_model(
    # headers
    modelId='anthropic.claude-v2:1',
    accept='application/json',
    contentType='application/json',

    # body as json string
    body=body,
)

Read **body** from the response.
Then, extract answer (located in: *response.body.completion*)

In [ ]:
body = json.loads(response["body"].read())
answer = body["completion"]
print("Model answer:", answer)

In [15]:
# try:
#     response = bedrock_client.invoke_model(
#         model_id = "anthropic.claude-v2:1",
#         accept = "application/json",
#         content_type = "application/json",
#         body=body,
#     )

#     response_body = json.loads(response.get("body").read())
#     outputText = response_body.get("results")[0].get("outputText")

# except botocore.exceptions.ClientError as error:
#     if error.response['Error']['Code'] == 'AccessDeniedException':
#         print(f"\x1b[41m{error.response['Error']['Message']}\
#         \nTo troubleshoot this error, please refer to the documentation: \x1b[0m\n"
#         )
#     else:
#         raise error

# # Executed cell:
# # NoCredentialsError: Unable to locate credentials

## Few Shot
https://github.com/ExamProCo/AWS-Examples/blob/main/bedrock/prompt-engineering/few-shot.ipynb

- model: amazon titan text express

In [ ]:
prompt_data = """
Tweet: "I hate it when my phone battery dies.”: Sentiment: Negative
Tweet: "My day has been great”: Sentiment: Positive
Tweet: "This is the link to the article”: Sentiment: Neutral
Tweet: "This new music video was incredible” Sentiment:
""" 

In [ ]:
body = json.dumps({
    "inputText": prompt_data,
}

In [ ]:
response = bedrock_client.invoke_model(
        modelId": 'amazon.titan-text-express-v1',
        accept": 'application/json',
        contentType": 'application/json',
        body=body,
    )

In [ ]:
body = json.loads(response.get("body").read())
answer = body["results"][0]["outputText"]
print(answer)

## CoT (Chain of Thought)
https://github.com/ExamProCo/AWS-Examples/blob/main/bedrock/prompt-engineering/chain-of-thought.ipynb

- model: anthropic claude3

In [ ]:
prompt_data = """
On a given week, the viewers for a TV channel were
Monday: 6500 viewers
Tuesday: 6400 viewers
Wednesday: 6300 viewers


Question: How many viewers can we expect on Friday?
Answer: Based on the numbers given and without any more information, there is a daily decrease of 100 viewers. If we assume this trend will continue during the following days, we can expect 6200 viewers on the next day that would be Thursday, and therefore 6100 viewers on the next day that would be Friday.

Question: How many viewers can we expect on Saturday? (Think Step-by-Step)
Answer: ?

Solve the anwser
"""

In [ ]:
body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 512,
    "messages": [{
        "role": "user",
        "content": [{
            "type": "text", 
            "text": prompt_data
            }]
        }],
    "temperature": 0.5,
    "top_p": 0.9,
}

In [ ]:
response = bedrock_client.invoke_model(
        modelId": 'anthropic.claude3-haiku-20240307-v1:0'
        accept": 'application/json',
        contentType": 'application/json',
        body=body,
    )

In [ ]:
body = json.loads(response.get("body").read())
answer = body["content"][0]["text"]
print("Model answer:", answer)